In [0]:
s%run ./connectionNotebook

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, round

In [0]:
catalog_name = 'adbrag'
target_schema_name = 'gold'
src_schema_name = 'silver'

In [0]:
silver_df = spark.table(f"{catalog_name}.{src_schema_name}.sales_silver")

In [0]:
gold_df = (
    silver_df
    .withColumn(
        "quantity_bucket",
        when(col("src_Quantity") <= 2, "Low")
        .when((col("src_Quantity") > 2) & (col("src_Quantity") <= 5), "Medium")
        .otherwise("High")
    )
    .withColumn(
        "revenue_bucket",
        when(col("src_TotalAmount") < 200, "Low Revenue")
        .when((col("src_TotalAmount") >= 200) & (col("src_TotalAmount") < 1000), "Medium Revenue")
        .otherwise("High Revenue")
    )
    .withColumn("gold_loaded_ts", current_timestamp())
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{target_schema_name}.sales_gold
(
    src_SaleID INT,
    src_Quantity INT,
    src_SaleDate DATE,
    src_TotalAmount DECIMAL(10,2),
    processed_ts TIMESTAMP,
    quantity_bucket STRING,
    revenue_bucket STRING,
    --avg_unit_amount DECIMAL(10,2),
    gold_loaded_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_SaleID)
""")

In [0]:
gold_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog_name}.{target_schema_name}.sales_gold")